# Markov Chain Model + Diagnostic

Notebook gọn nhẹ để:
1. Load data
2. Build transition matrices
3. Calibrate K values
4. Forecast
5. **🔍 DIAGNOSTIC: Tại sao DEL tăng liên tục?**

---

## 0️⃣ SETUP

In [ ]:
# Setup paths
import sys
from pathlib import Path
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Import modules
from src.config import CFG, BUCKETS_CANON, BUCKETS_30P, BUCKETS_90P
from src.config import parse_date_column, create_segment_columns, SEGMENT_COLS
from src.data_loader import load_data
from src.rollrate.transition import compute_transition_by_mob
from src.rollrate.lifecycle import get_actual_all_vintages_amount
from src.rollrate.calibration_kmob import (
    fit_k_raw, smooth_k, fit_alpha,
    forecast_all_vintages_partial_step,
)

print("✅ Import thành công")

## 1️⃣ LOAD DATA

In [ ]:
# ========== CẤU HÌNH ==========
DATA_PATH = 'C:/Users/User/Projection_PB/Projection_pb/ETB_Parquet_YYYYMM'  # 🔥 Thay đổi path của bạn
MAX_MOB = 36  # Forecast đến MOB 36
# ==============================

print("📂 Loading data...")
df_raw = load_data(DATA_PATH)
df_raw['DISBURSAL_DATE'] = parse_date_column(df_raw['DISBURSAL_DATE'])
df_raw = create_segment_columns(df_raw)

print(f"\n✅ Data loaded:")
print(f"   Rows: {len(df_raw):,}")
print(f"   Loans: {df_raw[CFG['loan']].nunique():,}")
print(f"   Products: {df_raw['PRODUCT_TYPE'].unique().tolist()}")
print(f"   Risk scores: {df_raw['RISK_SCORE'].nunique()} unique")
print(f"   Date range: {df_raw['CUTOFF_DATE'].min()} to {df_raw['CUTOFF_DATE'].max()}")

## 2️⃣ BUILD TRANSITION MATRICES

In [ ]:
print("🔨 Building transition matrices...")
matrices_by_mob, parent_fallback = compute_transition_by_mob(df_raw)

print(f"\n✅ Transition matrices built:")
print(f"   Products: {len(matrices_by_mob)}")
print(f"   Total matrices: {sum(len(m) for m in matrices_by_mob.values())}")
print(f"   Parent fallback: {len(parent_fallback)} cohorts")

## 3️⃣ CALIBRATE K VALUES

In [ ]:
print("🔨 Calibrating K values...")

# Get actual results
actual_results = get_actual_all_vintages_amount(df_raw)
print(f"   Actual results: {len(actual_results)} cohorts")

# Calculate DISB_TOTAL by vintage
loan_disb = df_raw.groupby(["PRODUCT_TYPE", "RISK_SCORE", CFG["orig_date"], CFG["loan"]])[CFG["disb"]].first()
disb_total_by_vintage = loan_disb.groupby(level=[0, 1, 2]).sum().to_dict()
print(f"   DISB_TOTAL: {len(disb_total_by_vintage)} cohorts")

# Fit K_raw with WLS Regularization
print("\n   Fitting K_raw (WLS Regularization)...")
k_raw_by_mob, weight_by_mob, df_k = fit_k_raw(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    include_co=True,
    denom_mode="disb",
    disb_total_by_vintage=disb_total_by_vintage,
    weight_mode="equal",
    method="wls_reg",
    lambda_k=1e-4,
    k_prior=0.0,
    min_obs=5,
    fallback_k=1.0,
    fallback_weight=0.0,
    return_detail=True,
)
print(f"   K_raw: {len(k_raw_by_mob)} MOBs")

# Smooth K
print("\n   Smoothing K...")
mob_min = min(k_raw_by_mob.keys()) if k_raw_by_mob else 0
mob_max = max(k_raw_by_mob.keys()) if k_raw_by_mob else 0
k_smooth_by_mob, _, _ = smooth_k(k_raw_by_mob, weight_by_mob, mob_min, mob_max)
print(f"   K_smooth: {len(k_smooth_by_mob)} MOBs")

# Fit alpha
print("\n   Fitting alpha...")
alpha, k_final_by_mob, df_alpha = fit_alpha(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    states=BUCKETS_CANON,
    s30_states=BUCKETS_30P,
    k_smooth_by_mob=k_smooth_by_mob,
    mob_target=min(MAX_MOB, mob_max) if mob_max else MAX_MOB,
    include_co=True,
    denom_mode="disb",
    disb_total_by_vintage=disb_total_by_vintage,
)

print(f"\n✅ Calibration complete:")
print(f"   Alpha: {alpha:.4f}")
print(f"   K_final: {len(k_final_by_mob)} MOBs")
print(f"   K range: [{min(k_final_by_mob.values()):.3f}, {max(k_final_by_mob.values()):.3f}]")

## 4️⃣ FORECAST

In [ ]:
print("🔮 Running forecast...")
forecast_results = forecast_all_vintages_partial_step(
    actual_results=actual_results,
    matrices_by_mob=matrices_by_mob,
    parent_fallback=parent_fallback,
    max_mob=MAX_MOB,
    k_by_mob=k_final_by_mob,
    states=BUCKETS_CANON,
)

print(f"\n✅ Forecast complete:")
print(f"   Cohorts: {len(forecast_results)}")
print(f"   Max MOB: {MAX_MOB}")

---

# 🔍 DIAGNOSTIC: TẠI SAO DEL TĂNG LIÊN TỤC?

Phần này sẽ chẩn đoán tại sao DEL curve tăng liên tục ở MOB cao thay vì flatten.

**Kiểm tra:**
1. ✅ K values ở MOB 25+
2. ✅ % cohorts dùng parent fallback ở MOB 24
3. ✅ So sánh P_24 vs Parent Fallback
4. ✅ Aggregation effect
5. ✅ Phân tích từng cohort

---

## 5️⃣ IMPORT DIAGNOSTIC SCRIPTS

In [ ]:
print("📥 Importing diagnostic scripts...")

try:
    from diagnose_why_increase_after_24 import diagnose_why_increase_after_24
    from check_p24_quality import check_p24_quality
    from diagnose_del_curve import diagnose_del_curve
    print("✅ Diagnostic scripts imported successfully")
except ImportError as e:
    print(f"❌ Error importing diagnostic scripts: {e}")
    print("   Make sure these files are in the project root:")
    print("   - diagnose_why_increase_after_24.py")
    print("   - check_p24_quality.py")
    print("   - diagnose_del_curve.py")

## 6️⃣ RUN MAIN DIAGNOSTIC

**Đây là cell quan trọng nhất!**

Cell này sẽ in ra báo cáo chi tiết với các chỉ báo ✅ hoặc ❌ cho từng vấn đề.

In [ ]:
print("🔍 Running comprehensive diagnostic...")
print("   This will check:")
print("   1. K values at MOB 25+")
print("   2. % cohorts using fallback at MOB 24")
print("   3. P_24 vs Parent Fallback comparison")
print("   4. Aggregation effects")
print("   5. Individual cohort analysis")
print("\n" + "="*80)

try:
    diagnose_why_increase_after_24(
        matrices_by_mob=matrices_by_mob,
        parent_fallback=parent_fallback,
        k_final_by_mob=k_final_by_mob,
        forecast_results=forecast_results,
        disb_total_by_vintage=disb_total_by_vintage,
        df_del_product=None  # Optional: aggregate product-level data
    )
except Exception as e:
    print(f"\n❌ Error running diagnostic: {e}")
    import traceback
    traceback.print_exc()

## 7️⃣ CHECK P_24 QUALITY (OPTIONAL)

Kiểm tra chi tiết chất lượng ma trận P_24 cho một cohort mẫu.

In [ ]:
print("\n" + "="*80)
print("🔍 Checking P_24 Quality for Sample Cohort")
print("="*80)

# Get a sample cohort to check
if matrices_by_mob:
    sample_product = list(matrices_by_mob.keys())[0]
    
    if 24 in matrices_by_mob[sample_product]:
        sample_score = list(matrices_by_mob[sample_product][24].keys())[0]
        
        print(f"\n📊 Analyzing: Product={sample_product}, Score={sample_score}")
        
        try:
            P_24, P_parent = check_p24_quality(
                matrices_by_mob=matrices_by_mob,
                parent_fallback=parent_fallback,
                product=sample_product,
                score=sample_score
            )
        except Exception as e:
            print(f"\n❌ Error checking P_24 quality: {e}")
    else:
        print("⚠️  MOB 24 not found in matrices_by_mob")
else:
    print("⚠️  matrices_by_mob is empty")

## 8️⃣ VISUALIZE DEL CURVE (OPTIONAL)

Vẽ biểu đồ DEL curve cho một cohort mẫu.

In [ ]:
print("\n" + "="*80)
print("📊 Visualizing DEL Curve for Sample Cohort")
print("="*80)

# Get a sample cohort with good data
if forecast_results:
    # Find a cohort with data at MOB 24
    sample_cohort = None
    for cohort_key, forecast_data in forecast_results.items():
        if 24 in forecast_data and len(forecast_data) > 10:
            sample_cohort = cohort_key
            break
    
    if sample_cohort:
        product, score, vintage = sample_cohort
        print(f"\n📊 Analyzing: Product={product}, Score={score}, Vintage={vintage}")
        
        try:
            diagnose_del_curve(
                matrices_by_mob=matrices_by_mob,
                parent_fallback=parent_fallback,
                k_final_by_mob=k_final_by_mob,
                forecast_results=forecast_results,
                disb_total_by_vintage=disb_total_by_vintage,
                product=product,
                score=score,
                vintage=vintage
            )
        except Exception as e:
            print(f"\n❌ Error visualizing DEL curve: {e}")
            import traceback
            traceback.print_exc()
    else:
        print("⚠️  No suitable cohort found for visualization")
else:
    print("⚠️  forecast_results is empty")

---

## 📋 DIAGNOSTIC SUMMARY & SOLUTIONS

Dựa vào kết quả diagnostic ở trên, áp dụng giải pháp phù hợp:

### ❌ Nếu K values quá cao (K > 0.9 ở MOB 25+)

**Giải pháp: Cap K ở MOB 25+**

```python
# Uncomment để áp dụng:
# for mob in range(25, 37):
#     if mob in k_final_by_mob:
#         k_final_by_mob[mob] = min(k_final_by_mob[mob], 0.3)
#     else:
#         k_final_by_mob[mob] = 0.3
# 
# # Re-run forecast
# forecast_results = forecast_all_vintages_partial_step(
#     actual_results=actual_results,
#     matrices_by_mob=matrices_by_mob,
#     parent_fallback=parent_fallback,
#     max_mob=MAX_MOB,
#     k_by_mob=k_final_by_mob,
#     states=BUCKETS_CANON,
# )
```

### ❌ Nếu nhiều cohorts dùng fallback ở MOB 24 (> 30%)

**Giải pháp A: Tăng MIN_OBS/MIN_EAD**

```python
# Sửa trong src/config.py:
# MIN_OBS = 200  # Thay vì 100
# MIN_EAD = 500  # Thay vì 100
# 
# Sau đó chạy lại từ Section 2 (Build Transition Matrices)
```

**Giải pháp B: Force dùng parent fallback cho MOB 25+**

Xem file `NEXT_STEPS_DIAGNOSIS.md` để biết cách sửa code.

### ❌ Nếu có aggregation issue

- Kiểm tra cohorts nào đang kéo DEL tăng
- Xem xét weights của từng cohort
- Có thể cần forecast riêng cho high-risk cohorts

---

## 📚 TÀI LIỆU THAM KHẢO

- **`NEXT_STEPS_DIAGNOSIS.md`** - Hướng dẫn chi tiết (English)
- **`HUONG_DAN_CHAY_DIAGNOSTIC.md`** - Hướng dẫn chi tiết (Tiếng Việt)
- **`DIAGNOSIS_CONTINUOUS_INCREASE.md`** - Lý thuyết và giải pháp
- **`CHECK_PARENT_FALLBACK_USAGE.md`** - Phân tích parent fallback

---

## 9️⃣ APPLY FIX (UNCOMMENT TO USE)

Sau khi xác định vấn đề từ diagnostic, uncomment giải pháp phù hợp dưới đây.

In [ ]:
# ============================================================
# SOLUTION 1: Cap K at MOB 25+
# ============================================================
# Uncomment nếu diagnostic cho thấy K quá cao

# print("🔧 Applying Solution 1: Capping K at MOB 25+")
# print("\nK values before:")
# for mob in range(24, 37):
#     print(f"  MOB {mob}: {k_final_by_mob.get(mob, 1.0):.3f}")

# # Cap K
# for mob in range(25, 37):
#     if mob in k_final_by_mob:
#         k_final_by_mob[mob] = min(k_final_by_mob[mob], 0.3)
#     else:
#         k_final_by_mob[mob] = 0.3

# print("\nK values after:")
# for mob in range(24, 37):
#     print(f"  MOB {mob}: {k_final_by_mob.get(mob, 1.0):.3f}")

# # Re-run forecast
# print("\n🔄 Re-running forecast with adjusted K...")
# forecast_results = forecast_all_vintages_partial_step(
#     actual_results=actual_results,
#     matrices_by_mob=matrices_by_mob,
#     parent_fallback=parent_fallback,
#     max_mob=MAX_MOB,
#     k_by_mob=k_final_by_mob,
#     states=BUCKETS_CANON,
# )
# print("✅ Forecast updated with adjusted K")

# # Re-run diagnostic to verify
# print("\n🔍 Re-running diagnostic to verify fix...")
# diagnose_why_increase_after_24(
#     matrices_by_mob=matrices_by_mob,
#     parent_fallback=parent_fallback,
#     k_final_by_mob=k_final_by_mob,
#     forecast_results=forecast_results,
#     disb_total_by_vintage=disb_total_by_vintage,
#     df_del_product=None
# )

print("💡 Uncomment the solution code above to apply the fix")
print("   Choose the solution based on your diagnostic results")

---

## ✅ HOÀN THÀNH!

Bạn đã chạy xong diagnostic. Các bước tiếp theo:

1. ✅ Đọc kết quả diagnostic ở Section 6
2. ✅ Xác định vấn đề (K cao? Fallback nhiều? Aggregation?)
3. ✅ Áp dụng giải pháp phù hợp ở Section 9
4. ✅ Re-run forecast và kiểm tra lại
5. ✅ Tiếp tục với lifecycle và export

**📚 Đọc thêm:**
- `HUONG_DAN_CHAY_DIAGNOSTIC.md` - Hướng dẫn đầy đủ bằng Tiếng Việt
- `NEXT_STEPS_DIAGNOSIS.md` - Detailed English guide

---